# Major Currency Behaviour Dashboard

This notebook tracks major currencies versus the U.S. dollar over the last 10 years using Yahoo Finance FX pairs.

## What is included

The dashboard covers a broad major-currency set:
- Euro (EUR)
- British Pound (GBP)
- Japanese Yen (JPY)
- Swiss Franc (CHF)
- Canadian Dollar (CAD)
- Australian Dollar (AUD)
- New Zealand Dollar (NZD)
- Chinese Yuan (CNY)

## Interpretation rule

All series are converted into the same direction: **USD per unit of foreign currency**.
That means:
- An upward move means the foreign currency is strengthening against the U.S. dollar.
- A downward move means the foreign currency is weakening against the U.S. dollar.

## Views in this notebook

1. Relative FX performance over 10 years
2. Regional composites
3. Rolling annualized volatility
4. Drawdown from peak
5. A compact JSON snapshot for downstream LLM use

In [1]:
%pip install -q pandas plotly yfinance


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import yfinance as yf

pair_definitions = [
    {"name": "Euro", "ticker": "EURUSD=X", "pair_label": "EUR/USD", "needs_inversion": False, "region": "Europe"},
    {"name": "British Pound", "ticker": "GBPUSD=X", "pair_label": "GBP/USD", "needs_inversion": False, "region": "Europe"},
    {"name": "Japanese Yen", "ticker": "JPY=X", "pair_label": "USD/JPY", "needs_inversion": True, "region": "Asia-Pacific"},
    {"name": "Swiss Franc", "ticker": "CHF=X", "pair_label": "USD/CHF", "needs_inversion": True, "region": "Europe"},
    {"name": "Canadian Dollar", "ticker": "CAD=X", "pair_label": "USD/CAD", "needs_inversion": True, "region": "North America"},
    {"name": "Australian Dollar", "ticker": "AUDUSD=X", "pair_label": "AUD/USD", "needs_inversion": False, "region": "Asia-Pacific"},
    {"name": "New Zealand Dollar", "ticker": "NZDUSD=X", "pair_label": "NZD/USD", "needs_inversion": False, "region": "Asia-Pacific"},
    {"name": "Chinese Yuan", "ticker": "CNY=X", "pair_label": "USD/CNY", "needs_inversion": True, "region": "Asia-Pacific"},
]

ticker_to_name = {item["ticker"]: item["name"] for item in pair_definitions}
name_to_region = {item["name"]: item["region"] for item in pair_definitions}
name_to_pair_label = {item["name"]: item["pair_label"] for item in pair_definitions}

raw = yf.download(
    tickers=list(ticker_to_name),
    period="10y",
    interval="1d",
    auto_adjust=False,
    progress=False,
    threads=False,
    group_by="column",
)

if raw.empty:
    raise ValueError("No FX data returned from Yahoo Finance.")

close = raw["Close"].rename(columns=ticker_to_name).sort_index()

available_names = [item["name"] for item in pair_definitions if item["name"] in close.columns]
missing_names = [item["name"] for item in pair_definitions if item["name"] not in close.columns]

if len(available_names) < 2:
    raise ValueError(f"Insufficient FX series returned. Available: {available_names}")

prices = close[available_names].copy()

for item in pair_definitions:
    name = item["name"]
    if name in prices.columns and item["needs_inversion"]:
        prices[name] = 1.0 / prices[name]

prices = prices.replace([np.inf, -np.inf], np.nan).dropna(how="all")
prices = prices.ffill().dropna(how="any")

normalized = prices.div(prices.iloc[0]).mul(100)
log_returns = np.log(prices / prices.shift(1)).dropna(how="all")
rolling_vol_63d = log_returns.rolling(63).std().mul(np.sqrt(252)).mul(100)
drawdown = prices.div(prices.cummax()).sub(1).mul(100)

lookback_windows = {
    "1M": 21,
    "3M": 63,
    "1Y": 252,
}

latest_snapshot = pd.DataFrame(index=prices.columns)
latest_snapshot["Last"] = prices.iloc[-1]
latest_snapshot["Since start %"] = normalized.iloc[-1] - 100
for label, window in lookback_windows.items():
    latest_snapshot[f"{label} return %"] = prices.pct_change(window).iloc[-1] * 100
latest_snapshot["Vol 63d %"] = rolling_vol_63d.iloc[-1]
latest_snapshot["Drawdown %"] = drawdown.iloc[-1]
latest_snapshot["Max drawdown %"] = drawdown.min()
latest_snapshot["Region"] = latest_snapshot.index.map(name_to_region)
latest_snapshot["Yahoo pair"] = latest_snapshot.index.map(name_to_pair_label)
latest_snapshot = latest_snapshot.sort_values("1Y return %", ascending=False)

region_members = {}
for name, region in name_to_region.items():
    if name in normalized.columns:
        region_members.setdefault(region, []).append(name)

region_relative = pd.DataFrame({
    region: normalized[members].mean(axis=1)
    for region, members in region_members.items()
})

region_snapshot = pd.DataFrame(index=region_relative.columns)
region_snapshot["Since start %"] = region_relative.iloc[-1] - 100
region_snapshot["3M return %"] = region_relative.pct_change(63).iloc[-1] * 100
region_snapshot["1Y return %"] = region_relative.pct_change(252).iloc[-1] * 100
region_snapshot = region_snapshot.sort_values("1Y return %", ascending=False)

leaders = latest_snapshot[["1Y return %", "3M return %", "Vol 63d %", "Drawdown %"]].head(3).round(2)
laggards = latest_snapshot[["1Y return %", "3M return %", "Vol 63d %", "Drawdown %"]].tail(3).round(2)
highest_vol = latest_snapshot[["Vol 63d %", "1Y return %", "Drawdown %"]].sort_values("Vol 63d %", ascending=False).head(3).round(2)
deepest_drawdowns = latest_snapshot[["Drawdown %", "Max drawdown %", "1Y return %"]].sort_values("Drawdown %").head(3).round(2)

breadth_positive_3m = int((latest_snapshot["3M return %"] > 0).sum())
breadth_positive_1y = int((latest_snapshot["1Y return %"] > 0).sum())
dispersion_3m = float(latest_snapshot["3M return %"].max() - latest_snapshot["3M return %"].min())

narrative_lines = [
    f"{latest_snapshot.index[0]} is the strongest major currency over 1 year at {latest_snapshot.iloc[0]['1Y return %']:.2f}% versus USD.",
    f"{latest_snapshot.index[-1]} is the weakest over 1 year at {latest_snapshot.iloc[-1]['1Y return %']:.2f}% versus USD.",
    f"{region_snapshot.index[0]} is the strongest regional bloc over 1 year at {region_snapshot.iloc[0]['1Y return %']:.2f}%.",
    f"Breadth is {breadth_positive_1y}/{len(latest_snapshot)} positive over 1 year and {breadth_positive_3m}/{len(latest_snapshot)} positive over 3 months.",
    f"Cross-sectional 3-month dispersion is {dispersion_3m:.2f} percentage points, highlighting how uneven recent FX moves have been.",
]

summary_payload = {
    "as_of": str(prices.index[-1].date()),
    "data_window": {
        "start": str(prices.index[0].date()),
        "end": str(prices.index[-1].date()),
        "frequency": "1d",
        "lookback": "10y",
    },
    "quote_convention": "All series converted to USD per unit of foreign currency so higher means foreign currency strength versus USD.",
    "available_currencies": available_names,
    "missing_currencies": missing_names,
    "leaders_1y": leaders.reset_index(names="currency").to_dict(orient="records"),
    "laggards_1y": laggards.reset_index(names="currency").to_dict(orient="records"),
    "highest_volatility": highest_vol.reset_index(names="currency").to_dict(orient="records"),
    "deepest_drawdowns": deepest_drawdowns.reset_index(names="currency").to_dict(orient="records"),
    "region_ranking_1y": region_snapshot.reset_index(names="region").round(2).to_dict(orient="records"),
    "breadth": {
        "positive_3m": breadth_positive_3m,
        "positive_1y": breadth_positive_1y,
        "universe_size": int(len(latest_snapshot)),
    },
    "dispersion_3m": round(dispersion_3m, 2),
    "narrative": narrative_lines,
}

fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=(
        "Normalized FX Strength (USD per foreign currency)",
        "Regional FX Composites",
        "63-Day Annualized Volatility (%)",
        "Drawdown From Peak (%)",
    ),
)

for name in normalized.columns:
    fig.add_trace(
        go.Scatter(
            x=normalized.index,
            y=normalized[name],
            name=name,
            mode="lines",
            hovertemplate=f"{name}<br>Date=%{{x|%Y-%m-%d}}<br>Index=%{{y:.2f}}<extra></extra>",
        ),
        row=1,
        col=1,
    )

for region in region_relative.columns:
    fig.add_trace(
        go.Scatter(
            x=region_relative.index,
            y=region_relative[region],
            name=f"{region} composite",
            mode="lines",
            hovertemplate=f"{region}<br>Date=%{{x|%Y-%m-%d}}<br>Index=%{{y:.2f}}<extra></extra>",
        ),
        row=2,
        col=1,
    )

for name in rolling_vol_63d.columns:
    fig.add_trace(
        go.Scatter(
            x=rolling_vol_63d.index,
            y=rolling_vol_63d[name],
            name=f"{name} vol",
            mode="lines",
            legendgroup=name,
            showlegend=False,
            hovertemplate=f"{name}<br>Date=%{{x|%Y-%m-%d}}<br>Vol=%{{y:.2f}}%%<extra></extra>",
        ),
        row=3,
        col=1,
    )

for name in drawdown.columns:
    fig.add_trace(
        go.Scatter(
            x=drawdown.index,
            y=drawdown[name],
            name=f"{name} drawdown",
            mode="lines",
            legendgroup=name,
            showlegend=False,
            hovertemplate=f"{name}<br>Date=%{{x|%Y-%m-%d}}<br>Drawdown=%{{y:.2f}}%%<extra></extra>",
        ),
        row=4,
        col=1,
    )

fig.update_yaxes(title_text="Index", row=1, col=1)
fig.update_yaxes(title_text="Index", row=2, col=1)
fig.update_yaxes(title_text="Vol %", row=3, col=1)
fig.update_yaxes(title_text="Drawdown %", row=4, col=1)

fig.update_layout(
    height=1400,
    width=1200,
    template="plotly_white",
    title={
        "text": "Major Currency Behaviour vs USD Over the Last 10 Years",
        "x": 0.5,
    },
    legend={
        "orientation": "h",
        "yanchor": "bottom",
        "y": 1.02,
        "xanchor": "center",
        "x": 0.5,
    },
)

fig.show()

print("Latest FX snapshot:")
display(latest_snapshot.round(2))

print("LLM summary:")
for line in narrative_lines:
    print(f"- {line}")

Latest FX snapshot:


,Last,Since start %,1M return %,3M return %,1Y return %,Vol 63d %,Drawdown %,Max drawdown %,Region,Yahoo pair
Ticker,,,,,,,,,,
Australian Dollar,0.72,-8.12,1.16,6.42,11.43,12.39,-11.74,-29.25,Asia-Pacific,AUD/USD
Chinese Yuan,0.15,-5.41,1.22,2.10,7.00,3.98,-8.06,-14.73,Asia-Pacific,USD/CNY
Swiss Franc,1.28,23.31,1.11,1.21,5.28,9.08,-2.16,-13.36,Europe,USD/CHF
Euro,1.18,3.65,1.70,0.40,3.20,7.88,-5.88,-23.29,Europe,EUR/USD
Canadian Dollar,0.73,-7.18,0.42,1.13,1.11,5.26,-11.99,-18.20,North America,USD/CAD
British Pound,1.35,-6.01,0.74,0.62,0.66,8.47,-8.55,-27.46,Europe,GBP/USD
New Zealand Dollar,0.59,-16.28,0.33,1.03,-1.41,11.24,-21.77,-26.67,Asia-Pacific,NZD/USD
Japanese Yen,0.01,-31.17,-0.50,-0.35,-10.42,9.14,-37.05,-38.19,Asia-Pacific,USD/JPY


LLM summary:
- Australian Dollar is the strongest major currency over 1 year at 11.43% versus USD.
- Japanese Yen is the weakest over 1 year at -10.42% versus USD.
- Europe is the strongest regional bloc over 1 year at 3.22%.
- Breadth is 6/8 positive over 1 year and 7/8 positive over 3 months.
- Cross-sectional 3-month dispersion is 6.77 percentage points, highlighting how uneven recent FX moves have been.


In [4]:
ranking_windows = ["1M return %", "3M return %", "1Y return %"]

ranking_frames = {}
for metric in ranking_windows:
    ranking_frames[metric] = (
        latest_snapshot[[metric, "Vol 63d %", "Drawdown %", "Region", "Yahoo pair"]]
        .sort_values(metric, ascending=False)
        .reset_index(names="currency")
    )
    ranking_frames[metric].index = ranking_frames[metric].index + 1
    ranking_frames[metric].index.name = "rank"

ranking_payload = {
    metric: frame.reset_index().to_dict(orient="records")
    for metric, frame in ranking_frames.items()
}

print("Currency rankings versus USD")
for metric, frame in ranking_frames.items():
    print(f"\n{metric}:")
    display(frame.round(2))

ranked_1y = ranking_frames["1Y return %"].reset_index()

rank_fig = go.Figure(
    go.Bar(
        x=ranked_1y["1Y return %"],
        y=ranked_1y["currency"],
        orientation="h",
        marker_color=np.where(ranked_1y["1Y return %"] >= 0, "#2a9d8f", "#e76f51"),
        customdata=ranked_1y[["rank", "Region", "Yahoo pair", "Vol 63d %", "Drawdown %"]],
        hovertemplate=(
            "Rank %{customdata[0]}<br>"
            "%{y}<br>"
            "1Y return=%{x:.2f}%<br>"
            "Region=%{customdata[1]}<br>"
            "Yahoo pair=%{customdata[2]}<br>"
            "Vol 63d=%{customdata[3]:.2f}%<br>"
            "Drawdown=%{customdata[4]:.2f}%<extra></extra>"
        ),
    )
)

rank_fig.update_layout(
    template="plotly_white",
    title={"text": "Major Currency Ranking by 1Y Return vs USD", "x": 0.5},
    xaxis_title="1Y return (%)",
    yaxis_title="Currency",
    height=500,
)
rank_fig.update_yaxes(autorange="reversed")
rank_fig.show()

print("Ranking JSON preview:")
print(json.dumps({"1Y return %": ranking_payload["1Y return %"]}, indent=2))

Currency rankings versus USD

1M return %:


,currency,1M return %,Vol 63d %,Drawdown %,Region,Yahoo pair
rank,,,,,,
1,Euro,1.70,7.88,-5.88,Europe,EUR/USD
2,Chinese Yuan,1.22,3.98,-8.06,Asia-Pacific,USD/CNY
3,Australian Dollar,1.16,12.39,-11.74,Asia-Pacific,AUD/USD
4,Swiss Franc,1.11,9.08,-2.16,Europe,USD/CHF
5,British Pound,0.74,8.47,-8.55,Europe,GBP/USD
6,Canadian Dollar,0.42,5.26,-11.99,North America,USD/CAD
7,New Zealand Dollar,0.33,11.24,-21.77,Asia-Pacific,NZD/USD
8,Japanese Yen,-0.50,9.14,-37.05,Asia-Pacific,USD/JPY



3M return %:


,currency,3M return %,Vol 63d %,Drawdown %,Region,Yahoo pair
rank,,,,,,
1,Australian Dollar,6.42,12.39,-11.74,Asia-Pacific,AUD/USD
2,Chinese Yuan,2.10,3.98,-8.06,Asia-Pacific,USD/CNY
3,Swiss Franc,1.21,9.08,-2.16,Europe,USD/CHF
4,Canadian Dollar,1.13,5.26,-11.99,North America,USD/CAD
5,New Zealand Dollar,1.03,11.24,-21.77,Asia-Pacific,NZD/USD
6,British Pound,0.62,8.47,-8.55,Europe,GBP/USD
7,Euro,0.40,7.88,-5.88,Europe,EUR/USD
8,Japanese Yen,-0.35,9.14,-37.05,Asia-Pacific,USD/JPY



1Y return %:


,currency,1Y return %,Vol 63d %,Drawdown %,Region,Yahoo pair
rank,,,,,,
1,Australian Dollar,11.43,12.39,-11.74,Asia-Pacific,AUD/USD
2,Chinese Yuan,7.00,3.98,-8.06,Asia-Pacific,USD/CNY
3,Swiss Franc,5.28,9.08,-2.16,Europe,USD/CHF
4,Euro,3.20,7.88,-5.88,Europe,EUR/USD
5,Canadian Dollar,1.11,5.26,-11.99,North America,USD/CAD
6,British Pound,0.66,8.47,-8.55,Europe,GBP/USD
7,New Zealand Dollar,-1.41,11.24,-21.77,Asia-Pacific,NZD/USD
8,Japanese Yen,-10.42,9.14,-37.05,Asia-Pacific,USD/JPY


Ranking JSON preview:
{
  "1Y return %": [
    {
      "rank": 1,
      "currency": "Australian Dollar",
      "1Y return %": 11.429500698051442,
      "Vol 63d %": 12.389101743986073,
      "Drawdown %": -11.744685475775585,
      "Region": "Asia-Pacific",
      "Yahoo pair": "AUD/USD"
    },
    {
      "rank": 2,
      "currency": "Chinese Yuan",
      "1Y return %": 7.001924220896205,
      "Vol 63d %": 3.9797294616154444,
      "Drawdown %": -8.055216213932393,
      "Region": "Asia-Pacific",
      "Yahoo pair": "USD/CNY"
    },
    {
      "rank": 3,
      "currency": "Swiss Franc",
      "1Y return %": 5.275366970023465,
      "Vol 63d %": 9.07648883124133,
      "Drawdown %": -2.158868792456581,
      "Region": "Europe",
      "Yahoo pair": "USD/CHF"
    },
    {
      "rank": 4,
      "currency": "Euro",
      "1Y return %": 3.201453856532499,
      "Vol 63d %": 7.882573511163812,
      "Drawdown %": -5.880148913357108,
      "Region": "Europe",
      "Yahoo pair": "EUR/USD"
 